In [5]:
# #start OMOP infrastucture build for vocab mapping
# !pip list
# #add needed packages
# !pip install sqlalchemy jupyter notebook

In [ ]:
r"""
Outline of steps
1 - clone repo 
        git clone https://github.com/paulnagy/DICOM2OMOP.git
2 - setup env
        pip install pandas pydicom sqlalchemy jupyter
3 - create OMOP db in SQlite: 
        python setup_omop_sqlite.py 
        vscode terminal to folder with .ipynb file
            cd "C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project"
            jupyter nbconvert --to notebook --execute --inplace setup_omop_sqlite.ipynb
4  - load DICOM vocab concepts from repo
        python load_dicom_concepts.py "DICOM2OMOP/files/OMOP CDM Staging/omop_table_staging_v3.csv"

        vscode terminal to cd "C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project"
        jupyter nbconvert --to notebook --execute --inplace load_dicom_concepts.ipynb --stdout
5 - add standard clinical concepts
        python add_standard_concepts.py
6 - run ETL pipeline on ToothFairy DICOMs
        python toothfairy_dicom_to_omop.py /path/to/toothfairy/dicom/
7 - query and demonstration 
        python query_omop_demo.py
8 - vocab analysis of missingness
        python vocab_mapping_analysis.py
"""

'\nOutline of steps\n1 - clone repo git clone https://github.com/paulnagy/DICOM2OMOP.git\n2 - setup env\n3 - create OMOP db in SQlite: vscode terminal to folder with .ipynb file\n    cd "C:\\Users\\julie\\AI agent courses\\dental_imaging_project_local\\DICOM2OMOP\\julie_project"\n    jupyter nbconvert --to notebook --execute --inplace setup_omop_sqlite.ipynb\n4  - load DICOM vocab concepts from repo\n5 - add standard clinical concepts\n6 - run ETL pipeline on ToothFairy DICOMs\n7 - query and demonstration \n8 - vocab analysis of missingness\n'

In [7]:
# 
# Create the OMOP CDM Core Tables (Minimal Set)
import sqlite3
from pathlib import Path


DB_PATH = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project\omop_dental_cbct_v1.db"

def create_omop_tables(db_path = DB_PATH):
    #create minimal OMOP CDM 54. tables for imaging extension demo
    conn = sqlite3.connect(db_path)

    #turn on foriegn key enforcement
    conn.execute("PRAGMA foriegn_keys=ON;")
    cur = conn.cursor()

    # ============================================================
    # VOCABULARY TABLES (needed to store DICOM concept mappings)
    # ============================================================
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS concept (
        concept_id          INTEGER PRIMARY KEY,
        concept_name        TEXT NOT NULL,
        domain_id           TEXT NOT NULL,
        vocabulary_id       TEXT NOT NULL,
        concept_class_id    TEXT NOT NULL,
        standard_concept    TEXT,
        concept_code        TEXT NOT NULL,
        valid_start_date    TEXT NOT NULL,
        valid_end_date      TEXT NOT NULL,
        invalid_reason      TEXT
    );
    """)
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS concept_relationship (
        concept_id_1        INTEGER NOT NULL,
        concept_id_2        INTEGER NOT NULL,
        relationship_id     TEXT NOT NULL,
        valid_start_date    TEXT NOT NULL,
        valid_end_date      TEXT NOT NULL,
        invalid_reason      TEXT
    );
    """)
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS vocabulary (
        vocabulary_id       TEXT PRIMARY KEY,
        vocabulary_name     TEXT NOT NULL,
        vocabulary_reference TEXT,
        vocabulary_version  TEXT,
        vocabulary_concept_id INTEGER
    );
    """)
    
    # ============================================================
    # CLINICAL DATA TABLES (minimal for demo)
    # ============================================================
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS person (
        person_id                   INTEGER PRIMARY KEY,
        gender_concept_id           INTEGER NOT NULL,
        year_of_birth               INTEGER NOT NULL,
        month_of_birth              INTEGER,
        day_of_birth                INTEGER,
        birth_datetime              TEXT,
        race_concept_id             INTEGER NOT NULL,
        ethnicity_concept_id        INTEGER NOT NULL,
        location_id                 INTEGER,
        provider_id                 INTEGER,
        care_site_id                INTEGER,
        person_source_value         TEXT,
        gender_source_value         TEXT,
        gender_source_concept_id    INTEGER,
        race_source_value           TEXT,
        race_source_concept_id      INTEGER,
        ethnicity_source_value      TEXT,
        ethnicity_source_concept_id INTEGER
    );
    """)

    # ============================================================
    # IMAGING EXTENSION TABLES (from Park et al. / DICOM2OMOP)
    # ============================================================
    """change from official version: 
    https://github.com/paulnagy/DICOM2OMOP/blob/main/dicom_standard_to_omop/build_imaging_extension.ipynb
    # Create Image_occurrence table
    # postgre sql does not have varchar(max) so we will use varchar(10000)
    ddl_statement = 
    CREATE TABLE adni.image_occurrence(
        image_occurrence_id integer NOT NULL,
        person_id integer NOT NULL,
        procedure_occurrence_id integer NOT NULL,
        visit_occurrence_id integer,
        anatomic_site_concept_id integer,
        wadors_uri varchar(10000),
        local_path varchar(10000),
        image_occurrence_date date NOT NULL,
        image_study_uid varchar(10000) NOT NULL,
        image_series_uid varchar(10000) NOT NULL,
        modality_concept_id integer NOT NULL
    );
    ddl_statement = 
    CREATE TABLE adni.image_feature(
    image_feature_id integer NOT NULL,
    person_id integer NOT NULL,
    image_occurrence_id integer NOT NULL,
    image_feature_event_field_concept_id integer,
    image_feature_event_id integer,
    image_feature_concept_id integer NOT NULL,
    image_feature_type_concept_id integer NOT NULL,
    image_finding_concept_id integer,
    image_finding_id integer,
    anatomic_site_concept_id integer,
    alg_system varchar(10000),
    alg_datetime timestamp
);
"""
#------------------------------------
    cur.execute("""
    CREATE TABLE IF NOT EXISTS image_occurrence (
        image_occurrence_id         INTEGER PRIMARY KEY AUTOINCREMENT,
        person_id                   INTEGER NOT NULL,
        -- procedure_occurrence_id integer NOT NULL,  --  is how to comment out in SQL
        -- visit_occurrence_id integer,
        anatomic_site_concept_id    INTEGER,
        -- wadors_uri varchar(10000),
        local_path TEXT,
        image_occurrence_date       TEXT,
        image_study_uid             TEXT,
        image_series_uid            TEXT,
        modality_concept_id         INTEGER,
                
        -- additions below: note image_occurrence_source_value not in omop standard, number_of_frames not in Toothfairy
        image_type                  TEXT,
        image_resolution_rows       INTEGER,
        image_resolution_columns    INTEGER,
        image_slice_thickness       REAL,
        number_of_frames            INTEGER,
                
        -- provenance fields
        image_occurrence_source_value TEXT,
        FOREIGN KEY (person_id) REFERENCES person(person_id)
    );
    """)

    #skip image_feature table until later

    conn.commit()
    print(f"OMOP tables created in {db_path}")

    #verify working - sqlite_master special hidden table in every SQLiite db, return names of tables in db
    cur.execute("SELECT name from sqlite_master WHERE type='table';")   #list of tuples, need first
    tables = [row[0] for row in cur.fetchall()]
    print(f"Tables: {', '.join(tables)}")

    conn.close()
    return db_path

if __name__ == '__main__':
    create_omop_tables()

    print('all done!')
 

OMOP tables created in C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project\omop_dental_cbct_v1.db
Tables: concept, concept_relationship, vocabulary, person, image_occurrence, sqlite_sequence
all done!


In [8]:
import pandas as pd

# Load your staging file
df = pd.read_csv(r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\files\OMOP CDM Staging\omop_table_staging_v3.csv")

# Check if the column exists first
if 'image_occurrence_source_value' in df.columns:
    # Count how many rows are NOT null or empty
    populated_count = df['image_occurrence_source_value'].notna().sum()
    
    print(f"Total rows: {len(df)}")
    print(f"Rows with a source value: {populated_count}")
    
    if populated_count > 0:
        print("\nHere are a few examples of what is in there:")
        # Drop the empty ones and show the first 5 unique values
        print(df['image_occurrence_source_value'].dropna().unique()[:5])
else:
    print("The column 'image_occurrence_source_value' does not exist in this CSV.")

The column 'image_occurrence_source_value' does not exist in this CSV.
